<a href="https://colab.research.google.com/github/AlnaIsana/Movierecommendation/blob/main/Movierecommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy scikit-learn requests

In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

In [3]:
import zipfile
import requests

url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

response = requests.get(url)

with open("movielens.zip", "wb") as f:
    f.write(response.content)

with zipfile.ZipFile("movielens.zip", "r") as zip_ref:
    zip_ref.extractall()

print("Dataset downloaded successfully!")

Dataset downloaded successfully!


In [4]:
movies = pd.read_csv('ml-latest-small/movies.csv')
ratings = pd.read_csv('ml-latest-small/ratings.csv')

print("Movies Shape:", movies.shape)
print("Ratings Shape:", ratings.shape)

movies.head()

Movies Shape: (9742, 3)
Ratings Shape: (100836, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
print(movies.info())

print("\nMissing Values:")
print(movies.isnull().sum())

movies.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB
None

Missing Values:
movieId    0
title      0
genres     0
dtype: int64


,movieId,title,genres
832,1093,"Doors, The (1991)",Drama
6213,45635,"Notorious Bettie Page, The (2005)",Drama
3972,5603,"Lavender Hill Mob, The (1951)",Comedy|Crime
8638,119155,Night at the Museum: Secret of the Tomb (2014),Adventure|Children|Comedy|Fantasy
7821,92694,Perfect Sense (2011),Drama|Romance|Sci-Fi


In [6]:
movies['genres'] = movies['genres'].fillna('')

movies[['title','genres']].head()

,title,genres
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy
2,Grumpier Old Men (1995),Comedy|Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance
4,Father of the Bride Part II (1995),Comedy


In [7]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies['genres'])

print(tfidf_matrix.shape)

(9742, 23)


In [8]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Similarity Matrix Shape:", cosine_sim.shape)

Similarity Matrix Shape: (9742, 9742)


In [9]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

indices.head()

,0
title,
Toy Story (1995),0
Jumanji (1995),1
Grumpier Old Men (1995),2
Waiting to Exhale (1995),3
Father of the Bride Part II (1995),4


In [10]:
def recommend_movies(movie_title, num_recommendations=10):

    if movie_title not in indices:
        return "Movie not found!"

    idx = indices[movie_title]

    similarity_scores = list(enumerate(cosine_sim[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:num_recommendations+1]

    movie_indices = [i[0] for i in similarity_scores]

    return movies[['title','genres']].iloc[movie_indices]

In [11]:
recommend_movies("Toy Story (1995)")

,title,genres
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
3000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
3568,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
6194,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy
6486,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy
6948,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy
7760,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy
8219,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy


In [15]:
def recommend_movies(movie_name, n=10):

    matches = movies[
        movies['title'].str.contains(
            movie_name,
            case=False,
            na=False
        )
    ]

    if matches.empty:
        return None

    selected_movie = matches.iloc[0]['title']

    print(f"\nMovie Found: {selected_movie}")

    idx = matches.index[0]

    similarity_scores = list(enumerate(cosine_sim[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )[1:n+1]

    movie_indices = [i[0] for i in similarity_scores]

    return movies.iloc[movie_indices][['title', 'genres']]

In [18]:
while True:

    movie_name = input("\nEnter Movie Name (or 'exit'): ").strip()

    if movie_name.lower() == "exit":
        break

    result = recommend_movies(movie_name)

    if result is None:
        print("Movie not found!")
    else:
        print("\nRecommended Movies:\n")
        print(result.to_string(index=False))


Enter Movie Name (or 'exit'): Jumanji

Movie Found: Jumanji (1995)

Recommended Movies:

                                                                                         title                     genres
                                                            Indian in the Cupboard, The (1995) Adventure|Children|Fantasy
                                                             NeverEnding Story III, The (1994) Adventure|Children|Fantasy
                                                               Escape to Witch Mountain (1975) Adventure|Children|Fantasy
                                                     Darby O'Gill and the Little People (1959) Adventure|Children|Fantasy
                                                                           Return to Oz (1985) Adventure|Children|Fantasy
                                                                 NeverEnding Story, The (1984) Adventure|Children|Fantasy
                                            NeverEnding 

In [19]:
movie_ratings = ratings.groupby('movieId').agg({
    'rating':['mean','count']
})

movie_ratings.columns = ['avg_rating','num_ratings']

movie_ratings.reset_index(inplace=True)

movie_stats = pd.merge(
    movies,
    movie_ratings,
    on='movieId'
)

movie_stats.head()

,movieId,title,genres,avg_rating,num_ratings
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.431818,110
2,3,Grumpier Old Men (1995),Comedy|Romance,3.259615,52
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.357143,7
4,5,Father of the Bride Part II (1995),Comedy,3.071429,49


In [20]:
top_movies = movie_stats[
    movie_stats['num_ratings'] > 50
].sort_values(
    by='avg_rating',
    ascending=False
)

top_movies[['title','avg_rating','num_ratings']].head(20)

,title,avg_rating,num_ratings
277,"Shawshank Redemption, The (1994)",4.429022,317
659,"Godfather, The (1972)",4.289062,192
2224,Fight Club (1999),4.272936,218
974,Cool Hand Luke (1967),4.271930,57
602,Dr. Strangelove or: How I Learned to Stop Worr...,4.268041,97
686,Rear Window (1954),4.261905,84
921,"Godfather: Part II, The (1974)",4.259690,129
6298,"Departed, The (2006)",4.252336,107
913,Goodfellas (1990),4.250000,126
694,Casablanca (1942),4.240000,100


In [21]:
def hybrid_recommend(movie_title, n=10):

    if movie_title not in indices:
        return "Movie not found"

    idx = indices[movie_title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(
        sim_scores,
        key=lambda x:x[1],
        reverse=True
    )[1:50]

    movie_indices = [i[0] for i in sim_scores]

    recommendations = movie_stats.iloc[movie_indices]

    recommendations = recommendations.sort_values(
        by='avg_rating',
        ascending=False
    )

    return recommendations[
        ['title','genres','avg_rating']
    ].head(n)

In [22]:
hybrid_recommend("Toy Story (1995)")


,title,genres,avg_rating
6194,Peaceful Warrior (2006),Drama,5.0
5490,Last Hurrah for Chivalry (Hao xia) (1979),Action|Drama,5.0
1706,"Impostors, The (1998)",Comedy,4.5
8900,Golmaal (2006),Children|Comedy,4.5
8927,The FP (2012),Comedy,4.5
5546,Days of Being Wild (A Fei jingjyuhn) (1990),Drama|Romance,4.5
9544,The Adventures of Sherlock Holmes and Doctor W...,(no genres listed),4.5
4424,I Capture the Castle (2003),Drama|Romance,4.5
2106,Oscar and Lucinda (a.k.a. Oscar & Lucinda) (1997),Drama|Romance,4.5
7917,Some Guy Who Kills People (2011),Comedy|Thriller,4.5


In [23]:
import pickle

pickle.dump(cosine_sim,
            open('movie_similarity.pkl','wb'))

pickle.dump(movies,
            open('movies.pkl','wb'))

print("Model Saved!")

Model Saved!


In [24]:
!pip install gradio pandas scikit-learn -q

In [25]:
import pandas as pd

movies = pd.read_csv("ml-latest-small/movies.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies["genres"] = movies["genres"].fillna("")

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(movies["genres"])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Similarity Matrix Created")

Similarity Matrix Created


In [27]:
def recommend(movie_name):

    matches = movies[
        movies["title"].str.contains(
            movie_name,
            case=False,
            na=False
        )
    ]

    if matches.empty:
        return pd.DataFrame(
            {"Result": ["Movie not found"]}
        )

    idx = matches.index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )[1:11]

    movie_indices = [i[0] for i in sim_scores]

    recommendations = movies.iloc[
        movie_indices
    ][["title", "genres"]]

    return recommendations

In [28]:
import gradio as gr

demo = gr.Interface(
    fn=recommend,
    inputs=gr.Textbox(
        label="Enter Movie Name",
        placeholder="Example: Toy Story"
    ),
    outputs=gr.Dataframe(),
    title="🎬 Movie Recommendation System",
    description="Content-Based Movie Recommender using MovieLens Dataset"
)

In [29]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5f11c0c050e3b37a4f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
